<a href="https://colab.research.google.com/github/sahithyaramesh2-oss/major-project-/blob/main/MSU_AVIS_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path

DATASET = Path("/content/drive/MyDrive/MSU_AVIS/MSU_AVIS")
PROBES = DATASET / "Probe_clips"
GALLERY = DATASET / "Gallery"

print("Dataset found:", DATASET.exists())
print("Probe folder found:", PROBES.exists())
print("Gallery folder found:", GALLERY.exists())

print("\nFolders inside dataset:")
for item in DATASET.iterdir():
    print(item.name)

Dataset found: True
Probe folder found: True
Gallery folder found: True

Folders inside dataset:
Probe_clips
Gallery
Eval_set


In [ ]:
from pathlib import Path

DATASET = Path("/content/drive/MyDrive/MSU_AVIS")

print("Dataset location:", DATASET)
print("\nItems inside dataset folder:")

for item in DATASET.iterdir():
    print("📁" if item.is_dir() else "📄", item.name)

Dataset location: /content/drive/MyDrive/MSU_AVIS

Items inside dataset folder:
📁 MSU_AVIS


In [ ]:

sessions = sorted(p for p in PROBES.iterdir() if p.is_dir())

print("Number of session folders:", len(sessions))
print("\nFirst 10 session folders:")

for folder in sessions[:10]:
    clips = list(folder.glob("*.mp4"))
    print(folder.name, "->", len(clips), "clips")


Number of session folders: 150

First 10 session folders:
S01_V1 -> 12 clips
S01_V2 -> 12 clips
S01_V3 -> 12 clips
S02_V1 -> 12 clips
S02_V2 -> 12 clips
S02_V3 -> 12 clips
S03_V1 -> 12 clips
S03_V2 -> 12 clips
S03_V3 -> 12 clips
S04_V1 -> 12 clips


In [ ]:

for folder in sorted(GALLERY.iterdir()):
    if folder.is_dir():
        files = list(folder.iterdir())
        print(f"\n{folder.name}: {len(files)} files")
        for file in files[:5]:
            print(" ", file.name)

print("\nSample probe clip:")
sample = next(PROBES.rglob("*.mp4"), None)
print(sample)



audio: 50 files
  G_A 1.mp4
  G_A 2.mp4
  G_A 3.mp4
  G_A 4.mp4
  G_A 5.mp4

face: 50 files
  G_F 1.mp4
  G_F 2.mp4
  G_F 3.mp4
  G_F 4.mp4
  G_F 5.mp4

Sample probe clip:
/content/drive/MyDrive/MSU_AVIS/MSU_AVIS/Probe_clips/S47_V2/clip_01.mp4


In [ ]:

!apt-get -qq install ffmpeg


In [ ]:

from pathlib import Path

DATASET = Path("/content/drive/MyDrive/MSU_AVIS/MSU_AVIS")
PROBES = DATASET / "Probe_clips"
GALLERY = DATASET / "Gallery"

print("Gallery exists:", GALLERY.exists())
print("Probes exist:", PROBES.exists())


Gallery exists: True
Probes exist: True


In [ ]:

import subprocess

samples = [
    GALLERY / "face" / "G_F 1.mp4",
    GALLERY / "audio" / "G_A 1.mp4",
    PROBES / "S01_V1" / "clip_01.mp4"
]

for file in samples:
    print("\nFile:", file.name)
    print("Exists:", file.exists())

    if file.exists():
        result = subprocess.run(
            [
                "ffprobe", "-v", "error",
                "-show_entries", "stream=codec_type,codec_name",
                "-of", "csv=p=0",
                str(file)
            ],
            capture_output=True,
            text=True
        )
        print(result.stdout or result.stderr)



File: G_F 1.mp4
Exists: True
h264,video
aac,audio


File: G_A 1.mp4
Exists: True
h264,video
aac,audio


File: clip_01.mp4
Exists: True
h264,video
aac,audio



In [ ]:

import subprocess
import json

samples = [
    GALLERY / "face" / "G_F 1.mp4",
    GALLERY / "audio" / "G_A 1.mp4",
    PROBES / "S01_V1" / "clip_01.mp4"
]

for file in samples:
    print("\n---", file.name, "---")

    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries",
            "format=duration:stream=codec_type,width,height,sample_rate,channels",
            "-of", "json",
            str(file)
        ],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        info = json.loads(result.stdout)
        print("Duration:", round(float(info["format"]["duration"]), 2), "seconds")
        for stream in info.get("streams", []):
            print(stream)
    else:
        print(result.stderr)



--- G_F 1.mp4 ---
Duration: 5.59 seconds
{'codec_type': 'video', 'width': 1920, 'height': 1080}
{'codec_type': 'audio', 'sample_rate': '48000', 'channels': 2}

--- G_A 1.mp4 ---
Duration: 16.01 seconds
{'codec_type': 'video', 'width': 1920, 'height': 1080}
{'codec_type': 'audio', 'sample_rate': '48000', 'channels': 1}

--- clip_01.mp4 ---
Duration: 5.0 seconds
{'codec_type': 'video', 'width': 1920, 'height': 1080}
{'codec_type': 'audio', 'sample_rate': '48000', 'channels': 2}


In [ ]:

from IPython.display import display, Video, Audio
import subprocess
from pathlib import Path

sample_face = GALLERY / "face" / "G_F 1.mp4"
sample_audio = GALLERY / "audio" / "G_A 1.mp4"
sample_probe = PROBES / "S01_V1" / "clip_01.mp4"

# Show a short preview of the gallery face file
print("1. Gallery face sample")
display(Video(str(sample_face), embed=True, width=480))

# Listen to the gallery audio sample
print("2. Gallery audio sample")
audio_path = "/tmp/gallery_audio.wav"

subprocess.run([
    "ffmpeg", "-y", "-i", str(sample_audio),
    "-vn", "-t", "8", "-ac", "1", "-ar", "16000",
    audio_path
], capture_output=True)

display(Audio(audio_path))

# Show a probe sample
print("3. Probe video sample")
display(Video(str(sample_probe), embed=True, width=480))


1. Gallery face sample


2. Gallery audio sample


3. Probe video sample


In [ ]:

print("Gallery face files:")
for f in sorted((GALLERY / "face").glob("*"))[:15]:
    print(f.name)

print("\nGallery audio files:")
for f in sorted((GALLERY / "audio").glob("*"))[:15]:
    print(f.name)

print("\nProbe session examples:")
for f in sorted(PROBES.iterdir())[:12]:
    print(f.name)


Gallery face files:
G_F 1.mp4
G_F 2.mp4
G_F 3.mp4
G_F 4.mp4
G_F 5.mp4
G_F 6.mp4
G_F 7.mp4
G_F 8.mp4
G_F 9.mp4
G_F10.mp4
G_F11.mp4
G_F12.mp4
G_F13.mp4
G_F14.mp4
G_F15.mp4

Gallery audio files:
G_A 1.mp4
G_A 2.mp4
G_A 3.mp4
G_A 4.mp4
G_A 5.mp4
G_A 6.mp4
G_A 7.mp4
G_A 8.mp4
G_A 9.mp4
G_A10.mp4
G_A11.mp4
G_A12.mp4
G_A13.mp4
G_A14.mp4
G_A15.mp4

Probe session examples:
S01_V1
S01_V2
S01_V3
S02_V1
S02_V2
S02_V3
S03_V1
S03_V2
S03_V3
S04_V1
S04_V2
S04_V3


In [ ]:

EVAL = DATASET / "Eval_set"

print("Evaluation folder exists:", EVAL.exists())

for item in sorted(EVAL.iterdir())[:10]:
    if item.is_dir():
        print("Folder:", item.name)
        clips = sorted(item.glob("*"))
        print("  Files:", len(clips))
        for f in clips[:3]:
            print(" ", f.name)
    else:
        print("File:", item.name)


Evaluation folder exists: True
Folder: vid01
  Files: 36
  clip_01.mp4
  clip_02.mp4
  clip_03.mp4
Folder: vid02
  Files: 36
  clip_01.mp4
  clip_02.mp4
  clip_03.mp4
Folder: vid03
  Files: 36
  clip_01.mp4
  clip_02.mp4
  clip_03.mp4
Folder: vid04
  Files: 36
  clip_01.mp4
  clip_02.mp4
  clip_03.mp4
Folder: vid05
  Files: 36
  clip_01.mp4
  clip_02.mp4
  clip_03.mp4
Folder: vid06
  Files: 37
  clip_01.mp4
  clip_02.mp4
  clip_03.mp4
Folder: vid07
  Files: 36
  clip_01.mp4
  clip_02.mp4
  clip_03.mp4
Folder: vid08
  Files: 36
  clip_01.mp4
  clip_02.mp4
  clip_03.mp4
Folder: vid09
  Files: 36
  clip_01.mp4
  clip_02.mp4
  clip_03.mp4
Folder: vid10
  Files: 36
  clip_01.mp4
  clip_02.mp4
  clip_03.mp4


In [ ]:

EVAL = DATASET / "Eval_set"

for folder_name in ["vid01", "vid02", "vid06"]:
    folder = EVAL / folder_name
    print(f"\n--- {folder_name} ---")

    if folder.exists():
        for file in sorted(folder.iterdir())[:12]:
            print(file.name)



--- vid01 ---
clip_01.mp4
clip_02.mp4
clip_03.mp4
clip_04.mp4
clip_05.mp4
clip_06.mp4
clip_07.mp4
clip_08.mp4
clip_09.mp4
clip_10.mp4
clip_11.mp4
clip_12.mp4

--- vid02 ---
clip_01.mp4
clip_02.mp4
clip_03.mp4
clip_04.mp4
clip_05.mp4
clip_06.mp4
clip_07.mp4
clip_08.mp4
clip_09.mp4
clip_10.mp4
clip_11.mp4
clip_12.mp4

--- vid06 ---
clip_01.mp4
clip_02.mp4
clip_03.mp4
clip_04.mp4
clip_05.mp4
clip_06.mp4
clip_07 (1).mp4
clip_07.mp4
clip_08.mp4
clip_09.mp4
clip_10.mp4
clip_11.mp4


In [ ]:

for item in sorted(EVAL.rglob("*")):
    if item.is_file() and item.suffix.lower() not in [".mp4", ".avi", ".mov", ".wav"]:
        print(item.relative_to(EVAL))


In [ ]:
from pathlib import Path
import re

print("Gallery face files:")
for f in sorted((GALLERY / "face").glob("*.mp4"))[:12]:
    print(f.name)

print("\nGallery audio files:")
for f in sorted((GALLERY / "audio").glob("*.mp4"))[:12]:
    print(f.name)

print("\nProbe folders:")
for f in sorted(PROBES.iterdir())[:12]:
    print(f.name)

print("\nProbe folder count:", sum(1 for f in PROBES.iterdir() if f.is_dir()))
print("Gallery face count:", sum(1 for f in (GALLERY / "face").glob("*.mp4")))
print("Gallery audio count:", sum(1 for f in (GALLERY / "audio").glob("*.mp4")))

Gallery face files:
G_F 1.mp4
G_F 2.mp4
G_F 3.mp4
G_F 4.mp4
G_F 5.mp4
G_F 6.mp4
G_F 7.mp4
G_F 8.mp4
G_F 9.mp4
G_F10.mp4
G_F11.mp4
G_F12.mp4

Gallery audio files:
G_A 1.mp4
G_A 2.mp4
G_A 3.mp4
G_A 4.mp4
G_A 5.mp4
G_A 6.mp4
G_A 7.mp4
G_A 8.mp4
G_A 9.mp4
G_A10.mp4
G_A11.mp4
G_A12.mp4

Probe folders:
S01_V1
S01_V2
S01_V3
S02_V1
S02_V2
S02_V3
S03_V1
S03_V2
S03_V3
S04_V1
S04_V2
S04_V3

Probe folder count: 150
Gallery face count: 50
Gallery audio count: 50


In [ ]:
from IPython.display import Video, display

print("Gallery face reference — possible subject 1")
display(Video(str(GALLERY / "face" / "G_F 1.mp4"), embed=True, width=400))

print("Probe video — subject 1, session 1")
display(Video(str(PROBES / "S01_V1" / "clip_01.mp4"), embed=True, width=400))

Gallery face reference — possible subject 1


Probe video — subject 1, session 1


In [ ]:


import sys
print("Python version:", sys.version)

try:
    import torch
    print("PyTorch version:", torch.__version__)
except Exception as e:
    print("PyTorch error:", e)



Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PyTorch version: 2.14.1+cpu


In [ ]:

%pip install -q insightface onnxruntime


In [ ]:

import cv2, re, time
import numpy as np
from pathlib import Path
from insightface.app import FaceAnalysis
from sklearn.metrics import roc_curve
from sklearn.metrics.pairwise import cosine_similarity

DATASET = Path("/content/drive/MyDrive/MSU_AVIS/MSU_AVIS")
GALLERY = DATASET / "Gallery" / "face"
PROBES = DATASET / "Probe_clips"

# Load pretrained face detector and embedding model
app = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])
app.prepare(ctx_id=-1, det_size=(320, 320))

def get_face_embedding(video_path):
    cap = cv2.VideoCapture(str(video_path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, n // 2))
    ok, frame = cap.read()
    cap.release()

    if not ok:
        return None

    faces = app.get(frame)
    if not faces:
        return None

    # Choose the largest detected face
    face = max(faces, key=lambda f:
               (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))
    emb = face.embedding.astype(np.float32)
    return emb / (np.linalg.norm(emb) + 1e-10)

# Extract gallery embeddings
gallery_embeddings = {}
for p in sorted(GALLERY.glob("*.mp4")):
    m = re.search(r"G_F\s*(\d+)", p.stem)
    if not m:
        continue
    sid = int(m.group(1))
    emb = get_face_embedding(p)
    if emb is not None:
        gallery_embeddings[sid] = emb

print("Gallery faces extracted:", len(gallery_embeddings))

# Sample 3 clips per person from session V1
results = []
start = time.time()

for folder in sorted(PROBES.glob("S*_V1")):
    m = re.match(r"S(\d+)_V1", folder.name)
    if not m:
        continue
    true_id = int(m.group(1))

    clips = sorted(folder.glob("*.mp4"))
    # Choose clips 1, 5 and 9 where available
    for idx in [0, 4, 8]:
        if idx >= len(clips):
            continue
        emb = get_face_embedding(clips[idx])
        if emb is None:
            continue

        ids = sorted(gallery_embeddings)
        scores = np.array([
            np.dot(emb, gallery_embeddings[sid]) for sid in ids
        ])
        results.append((true_id, ids, scores))

print("Probe clips successfully processed:", len(results))
print("Processing seconds:", round(time.time() - start, 1))

if not results:
    print("No usable probe faces found. Stop here and send me this output.")
else:
    # Identification accuracy
    correct = 0
    verification_labels, verification_scores = [], []

    for true_id, ids, scores in results:
        predicted_id = ids[int(np.argmax(scores))]
        correct += (predicted_id == true_id)

        for sid, score in zip(ids, scores):
            verification_labels.append(int(sid == true_id))
            verification_scores.append(float(score))

    accuracy = 100 * correct / len(results)

    # Verification: genuine vs impostor similarity scores
    y = np.array(verification_labels)
    s = np.array(verification_scores)
    fpr, tpr, thresholds = roc_curve(y, s)
    fnr = 1 - tpr
    k = int(np.nanargmin(np.abs(fpr - fnr)))
    eer = (fpr[k] + fnr[k]) / 2

    print("\n--- FACE-ONLY BASELINE ---")
    print(f"Identification accuracy: {accuracy:.2f}% ({correct}/{len(results)})")
    print(f"FAR near EER threshold: {fpr[k]*100:.2f}%")
    print(f"FRR near EER threshold: {fnr[k]*100:.2f}%")
    print(f"EER: {eer*100:.2f}%")
    print(f"Similarity threshold: {thresholds[k]:.4f}")


In [ ]:
%pip install -q speechbrain

In [ ]:

import os, re, subprocess, time
import numpy as np
import torch
from pathlib import Path
from speechbrain.inference.speaker import EncoderClassifier
from sklearn.metrics import roc_curve

DATASET = Path("/content/drive/MyDrive/MSU_AVIS/MSU_AVIS")
GALLERY_AUDIO = DATASET / "Gallery" / "audio"
PROBES = DATASET / "Probe_clips"
TMP = Path("/content/voice_baseline_tmp")
TMP.mkdir(exist_ok=True)

# Load pretrained speaker-recognition model
model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/pretrained_speaker_model",
    run_opts={"device": "cuda" if torch.cuda.is_available() else "cpu"}
)

def extract_audio(video_path, output_path):
    # Extract mono audio at 16 kHz from the video
    subprocess.run([
        "ffmpeg", "-y", "-v", "error",
        "-i", str(video_path),
        "-vn", "-ac", "1", "-ar", "16000",
        str(output_path)
    ], check=True)

def get_voice_embedding(video_path):
    wav_path = TMP / (video_path.stem.replace(" ", "_") + "_audio.wav")
    try:
        extract_audio(video_path, wav_path)
        signal = model.load_audio(str(wav_path))
        signal = signal.unsqueeze(0)
        with torch.no_grad():
            emb = model.encode_batch(signal).squeeze().cpu().numpy()
        emb = emb.astype(np.float32).flatten()
        return emb / (np.linalg.norm(emb) + 1e-10)
    except Exception as e:
        print("Skipped:", video_path.name, "|", str(e)[:120])
        return None

# Gallery speaker embeddings
gallery_embeddings = {}
for p in sorted(GALLERY_AUDIO.glob("*.mp4")):
    match = re.search(r"G_A\s*(\d+)", p.stem)
    if not match:
        continue
    sid = int(match.group(1))
    emb = get_voice_embedding(p)
    if emb is not None:
        gallery_embeddings[sid] = emb

print("Gallery voice embeddings:", len(gallery_embeddings))

# Probe clips: 3 selected clips from each V1 folder
results = []
start = time.time()

for folder in sorted(PROBES.glob("S*_V1")):
    match = re.match(r"S(\d+)_V1", folder.name)
    if not match:
        continue
    true_id = int(match.group(1))
    clips = sorted(folder.glob("*.mp4"))

    for idx in [0, 4, 8]:
        if idx >= len(clips):
            continue
        emb = get_voice_embedding(clips[idx])
        if emb is not None:
            results.append((true_id, emb))

print("Probe voice clips processed:", len(results))
print("Processing seconds:", round(time.time() - start, 1))

if not gallery_embeddings or not results:
    print("Not enough embeddings. Check the output above.")
else:
    ids = sorted(gallery_embeddings)
    correct = 0
    y_true, y_score = [], []

    for true_id, emb in results:
        scores = np.array([
            np.dot(emb, gallery_embeddings[sid]) for sid in ids
        ])
        correct += (ids[int(np.argmax(scores))] == true_id)

        for sid, score in zip(ids, scores):
            y_true.append(int(sid == true_id))
            y_score.append(float(score))

    accuracy = 100 * correct / len(results)
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    k = int(np.nanargmin(np.abs(fpr - fnr)))
    eer = (fpr[k] + fnr[k]) / 2

    print("\n--- VOICE-ONLY BASELINE ---")
    print(f"Identification accuracy: {accuracy:.2f}% ({correct}/{len(results)})")
    print(f"FAR near EER threshold: {fpr[k]*100:.2f}%")
    print(f"FRR near EER threshold: {fnr[k]*100:.2f}%")
    print(f"EER: {eer*100:.2f}%")
    print(f"Similarity threshold: {thresholds[k]:.4f}")


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


hyperparams.yaml:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


embedding_model.ckpt: reconstructing file:   0%|          |  0.00B / 83.3MB            

embedding_model.ckpt: downloading bytes:           |  0.00B            

INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


... (4 outputs left out)

INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


label_encoder.txt:   0%|          | 0.00/129k [00:00<?, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Gallery voice embeddings: 50
Probe voice clips processed: 150
Processing seconds: 165.2

--- VOICE-ONLY BASELINE ---
Identification accuracy: 56.00% (84/150)
FAR near EER threshold: 30.00%
FRR near EER threshold: 26.67%
EER: 28.33%
Similarity threshold: 0.1564


In [ ]:

# Face + voice fusion baseline
# Requires the previous face and voice cells to have run successfully.

import re, time
import numpy as np
from sklearn.metrics import roc_curve

# Rebuild face gallery embeddings (the previous voice cell reused the same variable)
face_gallery = {}
for p in sorted(GALLERY.glob("*.mp4")):
    m = re.search(r"G_F\s*(\d+)", p.stem)
    if m:
        sid = int(m.group(1))
        emb = get_face_embedding(p)
        if emb is not None:
            face_gallery[sid] = emb

# Voice gallery_embeddings already contains the voice references
common_ids = sorted(set(face_gallery) & set(gallery_embeddings))
print("Common gallery identities:", len(common_ids))

fusion_results = []
start = time.time()

for folder in sorted(PROBES.glob("S*_V1")):
    m = re.match(r"S(\d+)_V1", folder.name)
    if not m:
        continue
    true_id = int(m.group(1))
    clips = sorted(folder.glob("*.mp4"))

    for idx in [0, 4, 8]:
        if idx >= len(clips):
            continue

        clip = clips[idx]
        face_emb = get_face_embedding(clip)
        voice_emb = get_voice_embedding(clip)

        if face_emb is None or voice_emb is None:
            continue

        # Compare each probe with every common gallery identity
        face_scores = np.array([
            np.dot(face_emb, face_gallery[sid]) for sid in common_ids
        ])
        voice_scores = np.array([
            np.dot(voice_emb, gallery_embeddings[sid]) for sid in common_ids
        ])

        # Standardize scores per probe, then combine equally
        face_z = (face_scores - face_scores.mean()) / (face_scores.std() + 1e-8)
        voice_z = (voice_scores - voice_scores.mean()) / (voice_scores.std() + 1e-8)
        combined = 0.5 * face_z + 0.5 * voice_z

        fusion_results.append((true_id, combined))

print("Probe clips with both modalities:", len(fusion_results))
print("Processing seconds:", round(time.time() - start, 1))

if not fusion_results:
    print("No clips had both usable face and voice features.")
else:
    correct = 0
    labels, scores_all = [], []

    for true_id, scores in fusion_results:
        predicted_id = common_ids[int(np.argmax(scores))]
        correct += int(predicted_id == true_id)

        for sid, score in zip(common_ids, scores):
            labels.append(int(sid == true_id))
            scores_all.append(float(score))

    acc = 100 * correct / len(fusion_results)
    fpr, tpr, thresholds = roc_curve(labels, scores_all)
    fnr = 1 - tpr
    k = int(np.argmin(np.abs(fpr - fnr)))
    eer = (fpr[k] + fnr[k]) / 2

    print("\n--- FACE + VOICE FUSION ---")
    print(f"Identification accuracy: {acc:.2f}% ({correct}/{len(fusion_results)})")
    print(f"FAR near EER threshold: {fpr[k]*100:.2f}%")
    print(f"FRR near EER threshold: {fnr[k]*100:.2f}%")
    print(f"EER: {eer*100:.2f}%")
    print(f"Combined-score threshold: {thresholds[k]:.4f}")


Common gallery identities: 50


KeyboardInterrupt: 

In [ ]:

from pathlib import Path
from collections import Counter

DATASET = Path("/content/drive/MyDrive/MSU_AVIS/MSU_AVIS")
PROBES = DATASET / "Probe_clips"

sessions = [p for p in PROBES.iterdir() if p.is_dir()]

print("Total session folders:", len(sessions))

session_types = Counter(
    p.name.split("_")[-1] for p in sessions
)

print("\nSession distribution:")
for name, count in sorted(session_types.items()):
    print(name, ":", count)

print("\nSample session folders:")
for folder in sorted(sessions)[:15]:
    print(folder.name, ":", len(list(folder.glob("*.mp4"))), "clips")


Total session folders: 150

Session distribution:
V1 : 50
V2 : 50
V3 : 50

Sample session folders:
S01_V1 : 12 clips
S01_V2 : 12 clips
S01_V3 : 12 clips
S02_V1 : 12 clips
S02_V2 : 12 clips
S02_V3 : 12 clips
S03_V1 : 12 clips
S03_V2 : 12 clips
S03_V3 : 12 clips
S04_V1 : 12 clips
S04_V2 : 13 clips
S04_V3 : 12 clips
S05_V1 : 12 clips
S05_V2 : 12 clips
S05_V3 : 13 clips


In [ ]:

from pathlib import Path

DATASET = Path("/content/drive/MyDrive/MSU_AVIS/MSU_AVIS")

for folder_name in ["S01_V1", "S01_V2", "S01_V3"]:
    folder = DATASET / "Probe_clips" / folder_name

    print(f"\n{folder_name}")
    for file in sorted(folder.iterdir()):
        if file.is_file():
            print(file.name)



S01_V1
clip_01.mp4
clip_02.mp4
clip_03.mp4
clip_04.mp4
clip_05.mp4
clip_06.mp4
clip_07.mp4
clip_08.mp4
clip_09.mp4
clip_10.mp4
clip_11.mp4
clip_12.mp4

S01_V2
clip_01.mp4
clip_02.mp4
clip_03.mp4
clip_04.mp4
clip_05.mp4
clip_06.mp4
clip_07.mp4
clip_08.mp4
clip_09.mp4
clip_10.mp4
clip_11.mp4
clip_12.mp4

S01_V3
clip_01.mp4
clip_02.mp4
clip_03.mp4
clip_04.mp4
clip_05.mp4
clip_06.mp4
clip_07.mp4
clip_08.mp4
clip_09.mp4
clip_10.mp4
clip_11.mp4
clip_12.mp4


In [ ]:

GALLERY = DATASET / "Gallery"

print("Gallery structure:")
for item in sorted(GALLERY.iterdir()):
    if item.is_dir():
        print(item.name, "->", len(list(item.iterdir())), "files")
    else:
        print(item.name)


Gallery structure:
audio -> 50 files
face -> 50 files


In [ ]:

from pathlib import Path

GALLERY = Path("/content/drive/MyDrive/MSU_AVIS/MSU_AVIS/Gallery")

for modality in ["face", "audio"]:
    folder = GALLERY / modality

    print(f"\n========== {modality.upper()} GALLERY ==========")

    files = sorted(
        f for f in folder.iterdir() if f.is_file()
    )

    print("Total files:", len(files))

    for file in files:
        print(file.name)



========== FACE GALLERY ==========
Total files: 50
G_F 1.mp4
G_F 2.mp4
G_F 3.mp4
G_F 4.mp4
G_F 5.mp4
G_F 6.mp4
G_F 7.mp4
G_F 8.mp4
G_F 9.mp4
G_F10.mp4
G_F11.mp4
G_F12.mp4
G_F13.mp4
G_F14.mp4
G_F15.mp4
G_F16.mp4
G_F17.mp4
G_F18.mp4
G_F19.mp4
G_F20.mp4
G_F21.mp4
G_F22.mp4
G_F23.mp4
G_F24.mp4
G_F25.mp4
G_F26.mp4
G_F27.mp4
G_F28.mp4
G_F29.mp4
G_F30.mp4
G_F31.mp4
G_F32.mp4
G_F33.mp4
G_F34.mp4
G_F35.mp4
G_F36.mp4
G_F37.mp4
G_F38.mp4
G_F39.mp4
G_F40.mp4
G_F41.mp4
G_F42.mp4
G_F43.mp4
G_F44.mp4
G_F45.mp4
G_F46.mp4
G_F47.mp4
G_F48.mp4
G_F49.mp4
G_F50.mp4

========== AUDIO GALLERY ==========
Total files: 50
G_A 1.mp4
G_A 2.mp4
G_A 3.mp4
G_A 4.mp4
G_A 5.mp4
G_A 6.mp4
G_A 7.mp4
G_A 8.mp4
G_A 9.mp4
G_A10.mp4
G_A11.mp4
G_A12.mp4
G_A13.mp4
G_A14.mp4
G_A15.mp4
G_A16.mp4
G_A17.mp4
G_A18.mp4
G_A19.mp4
G_A20.mp4
G_A21.mp4
G_A22.mp4
G_A23.mp4
G_A24.mp4
G_A25.mp4
G_A26.mp4
G_A27.mp4
G_A28.mp4
G_A29.mp4
G_A30.mp4
G_A31.mp4
G_A32.mp4
G_A33.mp4
G_A34.mp4
G_A35.mp4
G_A36.mp4
G_A37.mp4
G_A38.mp4
G_A39.mp4
G_A40

In [ ]:
!pip install scikit-learn joblib

In [ ]:
%whos

Variable              Type                 Data/Info
----------------------------------------------------
Audio                 type                 <class 'IPython.lib.display.Audio'>
Counter               type                 <class 'collections.Counter'>
DATASET               PosixPath            /content/drive/MyDrive/MSU_AVIS/MSU_AVIS
EVAL                  PosixPath            /content/drive/MyDrive/MSU_AVIS/MSU_AVIS/Eval_set
EncoderClassifier     type                 <class 'speechbrain.infer<...>fiers.EncoderClassifier'>
FaceAnalysis          type                 <class 'insightface.app.f<...>e_analysis.FaceAnalysis'>
GALLERY               PosixPath            /content/drive/MyDrive/MSU_AVIS/MSU_AVIS/Gallery
GALLERY_AUDIO         PosixPath            /content/drive/MyDrive/MS<...>IS/MSU_AVIS/Gallery/audio
PROBES                PosixPath            /content/drive/MyDrive/MS<...>AVIS/MSU_AVIS/Probe_clips
Path                  type                 <class 'pathlib._local.Path'>
TMP 

In [ ]:
print("FACE GALLERY:")
print("Keys:", list(face_gallery.keys())[:5])

print("\nVOICE/GALLERY EMBEDDINGS:")
print("Keys:", list(gallery_embeddings.keys())[:5])

print("\nARRAY SHAPES:")
for name in [
    "face_scores", "voice_scores", "combined",
    "face_z", "voice_z", "scores_all",
    "labels", "y_true", "y_score"
]:
    obj = globals().get(name)
    if obj is not None:
        print(name, ":", np.shape(obj))

print("\nSAMPLE LABELS:", labels[:15])
print("SAMPLE SCORES:", scores_all[:15])

FACE GALLERY:
Keys: [1, 2, 3, 4, 5]

VOICE/GALLERY EMBEDDINGS:
Keys: [1, 2, 3, 4, 5]

ARRAY SHAPES:
face_scores : (50,)
voice_scores : (50,)
combined : (50,)
face_z : (50,)
voice_z : (50,)
scores_all : (5150,)
labels : (5150,)
y_true : (7500,)
y_score : (7500,)

SAMPLE LABELS: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
SAMPLE SCORES: [2.599250316619873, -1.1807670593261719, -0.6460000276565552, -0.682754635810852, 0.20066721737384796, -0.12309609353542328, -0.3788527250289917, 0.183681920170784, 0.38781440258026123, -0.14473950862884521, 0.9097610712051392, 1.286684513092041, 0.4829021096229553, 0.17188572883605957, -0.09780372679233551]


In [ ]:
print("========== FUSION RESULTS ==========")
print("Count:", len(fusion_results))
print("First item:", fusion_results[0])
print("Item type:", type(fusion_results[0]))

print("\n========== RESULTS ==========")
print("Count:", len(results))
print("First item:", results[0])

print("\n========== SCORE/LABEL COUNTS ==========")
print("Scores:", len(scores_all))
print("Labels:", len(labels))
print("Genuine:", sum(labels))
print("Impostor:", len(labels) - sum(labels))

========== FUSION RESULTS ==========
Count: 94
First item: (1, array([ 2.59925032e+00, -1.18076706e+00, -6.46000028e-01, -6.82754636e-01,
        2.00667217e-01, -1.23096094e-01, -3.78852725e-01,  1.83681920e-01,
        3.87814403e-01, -1.44739509e-01,  9.09761071e-01,  1.28668451e+00,
        4.82902110e-01,  1.71885729e-01, -9.78037268e-02,  1.77774191e+00,
        7.56380796e-01, -2.28134915e-01,  2.29335934e-01, -2.36500353e-01,
       -7.58152544e-01,  3.40761006e-01,  1.18354440e-01, -2.12197691e-01,
        1.60038769e-01,  4.03967440e-01, -3.21239740e-01,  2.20395535e-01,
        3.75129521e-01, -8.05116296e-02, -1.08190680e+00,  5.71318388e-01,
       -2.26521254e-01, -1.13195932e+00,  2.43023157e-01, -1.18940187e+00,
        3.65864962e-01, -3.39956969e-01, -1.05114996e+00, -1.09949887e+00,
        4.58492160e-01, -6.99037611e-01, -6.18956625e-01,  7.95502961e-02,
       -4.00769711e-01,  8.05139482e-01,  1.55943185e-02, -4.75720763e-01,
        2.59472400e-01,  2.42233276e-

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_curve
)

# Prepare data
X = np.array(scores_all).reshape(-1, 1)
y = np.array(labels)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Train RBF SVM
combined_svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced"
)

combined_svm.fit(X_train, y_train)

# Predictions
y_pred = combined_svm.predict(X_test)
y_decision = combined_svm.decision_function(X_test)

# Metrics
tn, fp, fn, tp = confusion_matrix(
    y_test, y_pred, labels=[0, 1]
).ravel()

accuracy = accuracy_score(y_test, y_pred)
far = fp / (fp + tn) if (fp + tn) else 0
frr = fn / (fn + tp) if (fn + tp) else 0

fpr, tpr, thresholds = roc_curve(y_test, y_decision)
fnr = 1 - tpr
eer_index = np.nanargmin(np.abs(fpr - fnr))
eer = (fpr[eer_index] + fnr[eer_index]) / 2

print("========== COMBINED RBF-SVM RESULTS ==========")
print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"FAR      : {far * 100:.2f}%")
print(f"FRR      : {frr * 100:.2f}%")
print(f"EER      : {eer * 100:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Impostor", "Genuine"],
    zero_division=0
))

========== COMBINED RBF-SVM RESULTS ==========
Accuracy : 93.17%
FAR      : 6.66%
FRR      : 15.38%
EER      : 13.38%

Confusion Matrix:
[[1178   84]
 [   4   22]]

Classification Report:
              precision    recall  f1-score   support

    Impostor       1.00      0.93      0.96      1262
     Genuine       0.21      0.85      0.33        26

    accuracy                           0.93      1288
   macro avg       0.60      0.89      0.65      1288
weighted avg       0.98      0.93      0.95      1288



In [ ]:
print("Face gallery embedding shape:",
      np.shape(face_gallery[1]))

print("Voice gallery embedding shape:",
      np.shape(gallery_embeddings[1]))

print("Fusion result count:", len(fusion_results))
print("Fusion feature shape:", fusion_results[0][1].shape)

print("Probe result count:", len(results))
print("Probe embedding shape:", results[0][1].shape)

Face gallery embedding shape: (512,)
Voice gallery embedding shape: (192,)
Fusion result count: 94
Fusion feature shape: (50,)
Probe result count: 150
Probe embedding shape: (192,)


In [ ]:
print("========== FUSION SAMPLE LABELS ==========")
print("Total:", len(fusion_results))
print("Genuine:", sum(x[0] for x in fusion_results))
print("Impostor:", len(fusion_results) - sum(x[0] for x in fusion_results))

print("\n========== VOICE SAMPLE LABELS ==========")
print("Total:", len(results))
print("Genuine:", sum(x[0] for x in results))
print("Impostor:", len(results) - sum(x[0] for x in results))

print("\n========== AVAILABLE EMBEDDING VARIABLES ==========")
for name, obj in globals().copy().items():
    if isinstance(obj, dict) and len(obj) > 0:
        first = next(iter(obj.values()))
        if isinstance(first, np.ndarray):
            print(name, len(obj), first.shape)

print("\n========== EMBEDDING FUNCTION DETAILS ==========")
import inspect
print(inspect.getsource(get_face_embedding))
print(inspect.getsource(get_voice_embedding))

========== FUSION SAMPLE LABELS ==========
Total: 94
Genuine: 2026
Impostor: -1932

========== VOICE SAMPLE LABELS ==========
Total: 150
Genuine: 3825
Impostor: -3675

========== AVAILABLE EMBEDDING VARIABLES ==========
gallery_embeddings 50 (192,)
face_gallery 50 (512,)

========== EMBEDDING FUNCTION DETAILS ==========
def get_face_embedding(video_path):
    cap = cv2.VideoCapture(str(video_path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, n // 2))
    ok, frame = cap.read()
    cap.release()

    if not ok:
        return None

    faces = app.get(frame)
    if not faces:
        return None

    # Choose the largest detected face
    face = max(faces, key=lambda f:
               (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))
    emb = face.embedding.astype(np.float32)
    return emb / (np.linalg.norm(emb) + 1e-10)

def get_voice_embedding(video_path):
    wav_path = TMP / (video_path.stem.replace(" ", "_") + "_audio.wav")
    try:
   

In [ ]:
import numpy as np
import cv2
from sklearn.svm import SVC
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    precision_score, recall_score, f1_score, roc_curve
)

# 1. Prepare probe sessions
sessions = sorted(
    [p for p in PROBES.iterdir() if p.is_dir()],
    key=lambda p: p.name
)

assert len(sessions) == len(results) == 150

face_X, voice_X, fusion_X = [], [], []
y, groups = [], []

gallery_ids = sorted(face_gallery.keys())

# 2. Generate verification pairs
for i, session in enumerate(sessions):

    true_id = int(session.name[1:3])
    result_id, voice_emb = results[i]

    assert int(result_id) == true_id, (
        f"Identity mismatch: {session.name} vs {result_id}"
    )

    clips = sorted(session.glob("*.mp4"))
    if not clips:
        continue

    face_emb = get_face_embedding(clips[0])

    if face_emb is None:
        print("Skipping face detection failure:", session.name)
        continue

    for gid in gallery_ids:

        f_score = float(np.dot(face_emb, face_gallery[gid]))
        v_score = float(np.dot(voice_emb, gallery_embeddings[gid]))

        face_X.append([f_score])
        voice_X.append([v_score])
        fusion_X.append([f_score, v_score])

        y.append(1 if true_id == gid else 0)
        groups.append(i)

# 3. Convert to arrays
X_face = np.asarray(face_X)
X_voice = np.asarray(voice_X)
X_fusion = np.asarray(fusion_X)

y = np.asarray(y)
groups = np.asarray(groups)

print("Total pairs:", len(y))
print("Genuine pairs:", np.sum(y == 1))
print("Impostor pairs:", np.sum(y == 0))

# 4. Split by probe session to avoid placing pairs
# from the same probe in both training and testing.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X_fusion, y, groups)
)

# 5. Train and evaluate all three models
datasets = {
    "Face-only": X_face,
    "Voice-only": X_voice,
    "Face + Voice": X_fusion
}

models = {}
all_results = {}

for name, X in datasets.items():

    svm = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced"
    )

    svm.fit(X[train_idx], y[train_idx])

    pred = svm.predict(X[test_idx])
    decision = svm.decision_function(X[test_idx])

    tn, fp, fn, tp = confusion_matrix(
        y[test_idx], pred, labels=[0, 1]
    ).ravel()

    far = fp / (fp + tn) if fp + tn else 0
    frr = fn / (fn + tp) if fn + tp else 0

    fpr, tpr, _ = roc_curve(y[test_idx], decision)
    fnr = 1 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2

    models[name] = svm

    all_results[name] = {
        "Accuracy": accuracy_score(y[test_idx], pred) * 100,
        "FAR": far * 100,
        "FRR": frr * 100,
        "EER": eer * 100,
        "Precision": precision_score(
            y[test_idx], pred, zero_division=0
        ) * 100,
        "Recall": recall_score(
            y[test_idx], pred, zero_division=0
        ) * 100,
        "F1-score": f1_score(
            y[test_idx], pred, zero_division=0
        ) * 100
    }

    print(f"\n========== {name} ==========")
    print("Confusion matrix:")
    print(confusion_matrix(y[test_idx], pred))
    for metric, value in all_results[name].items():
        print(f"{metric}: {value:.2f}%")

# 6. Final comparison
import pandas as pd

comparison = pd.DataFrame(all_results).T
print("\n========== MODEL COMPARISON ==========")
display(comparison.round(2))

Skipping face detection failure: S02_V2
Skipping face detection failure: S03_V3
Skipping face detection failure: S04_V3
Skipping face detection failure: S07_V2
Skipping face detection failure: S08_V1
Skipping face detection failure: S08_V3
Skipping face detection failure: S11_V3
Skipping face detection failure: S13_V1
Skipping face detection failure: S14_V1
Skipping face detection failure: S14_V2
Skipping face detection failure: S16_V2
Skipping face detection failure: S17_V1
Skipping face detection failure: S17_V3
Skipping face detection failure: S18_V1
Skipping face detection failure: S18_V3
Skipping face detection failure: S19_V2
Skipping face detection failure: S21_V2
Skipping face detection failure: S23_V3
Skipping face detection failure: S24_V1
Skipping face detection failure: S24_V3
Skipping face detection failure: S25_V1
Skipping face detection failure: S27_V1
Skipping face detection failure: S29_V1
Skipping face detection failure: S32_V1
Skipping face detection failure: S33_V2


,Accuracy,FAR,FRR,EER,Precision,Recall,F1-score
Face-only,98.34,1.34,17.24,9.01,55.81,82.76,66.67
Voice-only,94.76,4.43,44.83,28.25,20.25,55.17,29.63
Face + Voice,97.79,2.11,6.90,4.12,47.37,93.10,62.79


In [ ]:
import os
import pandas as pd

os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

comparison.to_csv(
    "results/svm_comparison.csv",
    index=True
)

print("Comparison results saved!")

Comparison results saved!


In [ ]:
import joblib

joblib.dump(models["Face-only"], "models/face_svm.pkl")
joblib.dump(models["Voice-only"], "models/voice_svm.pkl")
joblib.dump(models["Face + Voice"], "models/fusion_svm.pkl")

print("All three SVM models saved!")

All three SVM models saved!


In [ ]:

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    precision_score, recall_score, f1_score, roc_curve
)

# 1. Prepare gallery identities
common_ids = sorted(set(face_gallery) & set(gallery_embeddings))

print("Common gallery identities:", len(common_ids))

# 2. Extract matching face and voice features
face_X, voice_X, fusion_X = [], [], []
y, groups = [], []

processed_clips = 0
skipped_clips = 0

for folder in sorted(PROBES.glob("S*_V1")):

    true_id = int(folder.name[1:3])

    if true_id not in common_ids:
        continue

    clips = sorted(folder.glob("*.mp4"))

    for idx in [0, 4, 8]:

        if idx >= len(clips):
            continue

        clip = clips[idx]

        # Both embeddings come from this exact same video
        face_emb = get_face_embedding(clip)
        voice_emb = get_voice_embedding(clip)

        if face_emb is None or voice_emb is None:
            skipped_clips += 1
            continue

        processed_clips += 1

        for gid in common_ids:

            face_score = float(np.dot(face_emb, face_gallery[gid]))
            voice_score = float(np.dot(voice_emb, gallery_embeddings[gid]))

            face_X.append([face_score])
            voice_X.append([voice_score])
            fusion_X.append([face_score, voice_score])

            y.append(int(true_id == gid))
            groups.append(true_id)

# 3. Prepare arrays
X_face = np.asarray(face_X)
X_voice = np.asarray(voice_X)
X_fusion = np.asarray(fusion_X)

y = np.asarray(y)
groups = np.asarray(groups)

print("\nProcessed clips:", processed_clips)
print("Skipped clips:", skipped_clips)
print("Total verification pairs:", len(y))
print("Genuine pairs:", int(np.sum(y == 1)))
print("Impostor pairs:", int(np.sum(y == 0)))

assert len(y) > 0, "No usable verification pairs found."
assert len(np.unique(groups)) >= 2, "Not enough subjects for splitting."

# 4. Split by SUBJECT, not by individual pairs
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X_fusion, y, groups)
)

print("\nTraining subjects:", len(np.unique(groups[train_idx])))
print("Testing subjects:", len(np.unique(groups[test_idx])))

# 5. Train and evaluate three RBF-SVM models
datasets = {
    "Face-only": X_face,
    "Voice-only": X_voice,
    "Face + Voice": X_fusion
}

models = {}
all_results = {}

for name, X in datasets.items():

    svm = make_pipeline(
        StandardScaler(),
        SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced"
        )
    )

    svm.fit(X[train_idx], y[train_idx])

    pred = svm.predict(X[test_idx])
    decision = svm.decision_function(X[test_idx])

    tn, fp, fn, tp = confusion_matrix(
        y[test_idx], pred, labels=[0, 1]
    ).ravel()

    far = fp / (fp + tn) if (fp + tn) else 0
    frr = fn / (fn + tp) if (fn + tp) else 0

    fpr, tpr, _ = roc_curve(y[test_idx], decision)
    fnr = 1 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2

    models[name] = svm

    all_results[name] = {
        "Accuracy": accuracy_score(y[test_idx], pred) * 100,
        "FAR": far * 100,
        "FRR": frr * 100,
        "EER": eer * 100,
        "Precision": precision_score(
            y[test_idx], pred, zero_division=0
        ) * 100,
        "Recall": recall_score(
            y[test_idx], pred, zero_division=0
        ) * 100,
        "F1-score": f1_score(
            y[test_idx], pred, zero_division=0
        ) * 100
    }

    print(f"\n========== {name} ==========")
    print("Confusion matrix:")
    print(confusion_matrix(y[test_idx], pred))

    for metric, value in all_results[name].items():
        print(f"{metric}: {value:.2f}%")

# 6. Save comparison and models
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

comparison = pd.DataFrame(all_results).T
comparison.to_csv("results/svm_comparison.csv")

joblib.dump(models, "models/svm_models.pkl")

print("\n========== FINAL COMPARISON ==========")
display(comparison.round(2))

print("\nSaved comparison and trained models.")


Common gallery identities: 50

Processed clips: 103
Skipped clips: 47
Total verification pairs: 5150
Genuine pairs: 103
Impostor pairs: 5047

Training subjects: 33
Testing subjects: 12

========== Face-only ==========
Confusion matrix:
[[1158   18]
 [   9   15]]
Accuracy: 97.75%
FAR: 1.53%
FRR: 37.50%
EER: 15.01%
Precision: 45.45%
Recall: 62.50%
F1-score: 52.63%

========== Voice-only ==========
Confusion matrix:
[[1136   40]
 [   7   17]]
Accuracy: 96.08%
FAR: 3.40%
FRR: 29.17%
EER: 19.52%
Precision: 29.82%
Recall: 70.83%
F1-score: 41.98%

========== Face + Voice ==========
Confusion matrix:
[[1156   20]
 [   2   22]]
Accuracy: 98.17%
FAR: 1.70%
FRR: 8.33%
EER: 3.02%
Precision: 52.38%
Recall: 91.67%
F1-score: 66.67%

========== FINAL COMPARISON ==========


,Accuracy,FAR,FRR,EER,Precision,Recall,F1-score
Face-only,97.75,1.53,37.50,15.01,45.45,62.50,52.63
Voice-only,96.08,3.40,29.17,19.52,29.82,70.83,41.98
Face + Voice,98.17,1.70,8.33,3.02,52.38,91.67,66.67



Saved comparison and trained models.
